# Adding a New Model to the FRAMES Training Pipeline

This tutorial walks through how to plug a new model into the pipeline. We'll use XGBoost as the worked example since it's already in the codebase — so everything here is real, copy-pasteable code, not pseudocode.

There are only three things you need to do:

1. Create a model wrapper class in `models/`
2. Register it in `models/factory.py`
3. Add a config section in `config.yaml`

`main.py` doesn't need to change at all.

## Step 1: Create the Model Wrapper

Create a new file in `src/model_training/models/` for your model. Below is the full XGBoost wrapper (`models/xgboost_model.py`) — this is the exact code that's already in the pipeline.

A few things to note before reading it:

- The class inherits from `BaseTabularModel`, which handles the DataFrame → numpy array extraction before calling `fit_model`. Use `BaseSequenceModel` instead if your model operates on sequences (like the LSTM).
- You **must** implement `fit_model`, `predict_model`, and `save_model`. `custom_func` is optional.
- `self.config` gives you access to the full config, `self.features` is the list of feature column names, and `self.model` is where you store the trained model object.

In [3]:
# src/model_training/models/xgboost_model.py  (existing file — shown in full)

import optuna
import mlflow
import mlflow.xgboost
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score
from models.base_model import BaseTabularModel
from custom_funcs.custom_plots import shap_explanations


class XGBoostWrapper(BaseTabularModel):
    def __init__(self, config, model_params, features):
        super().__init__(config, model_params, features)
        # search_space comes from tabular.models.xgboost.search_space in config.yaml
        self.search_space = model_params.get('search_space', {})


    # turns the [min, max, type] entries in config.yaml into the Optuna call.
    def _suggest_param(self, trial, name, space):
        low, high, p_type = space
        if p_type == 'int':
            return trial.suggest_int(name, low, high)
        elif p_type == 'int_log':
            return trial.suggest_int(name, low, high, log=True)
        elif p_type == 'float':
            return trial.suggest_float(name, low, high)
        elif p_type == 'log':
            return trial.suggest_float(name, low, high, log=True)
        else:
            raise ValueError(f"param type '{p_type}' not understood.")

    # REQUIRED: fit_model
    # Receives data from BaseTabularModel.
    # Must set self.model to the trained model object.
    def fit_model(self, X_train, y_train, X_val, y_val):
        opt_config = self.config.get('tabular', {}).get('optimisation', {})
        sys_config = self.config['system']
        rnd_state = self.config['experiment']['random_state']

        # Optuna objective funciton
        def objective(trial):
            kwargs = {}
            for p_name, p_space in self.search_space.items():
                kwargs[p_name] = self._suggest_param(trial, p_name, p_space)

            kwargs['random_state'] = rnd_state
            kwargs['n_jobs'] = sys_config['n_jobs']
            kwargs.update({'objective': 'binary:logistic', 'eval_metric': 'aucpr'})

            model = xgb.XGBClassifier(**kwargs)
            model.set_params(early_stopping_rounds=30)
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
            preds_val = model.predict_proba(X_val)[:, 1]
            return average_precision_score(y_val, preds_val)

        # Run the HPO search
        study = optuna.create_study(direction="maximize", study_name="sepsis-xgboost-tuning")
        study.optimize(objective, n_trials=opt_config.get('num_trials', 20), show_progress_bar=True)

        # Retrain on the full train set with the best hyperparameters found
        best_kwargs = study.best_params.copy()
        best_kwargs['random_state'] = rnd_state
        best_kwargs['n_jobs'] = sys_config['n_jobs']
        best_kwargs.update({'objective': 'binary:logistic', 'eval_metric': 'aucpr'})

        mlflow.log_params(best_kwargs)
        mlflow.log_metric("best_val_auprc", study.best_value)

        self.model = xgb.XGBClassifier(**best_kwargs)
        self.model.set_params(early_stopping_rounds=30)
        self.model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        if hasattr(self.model, 'best_iteration'):
            mlflow.log_metric("best_iteration", self.model.best_iteration)


    # REQUIRED: predict_model
    # Must return an array of probabilities.
    def predict_model(self, X_test):
        return self.model.predict_proba(X_test)[:, 1]

    # REQUIRED: save_model
    # Save the model to MLflow.
    def save_model(self, model_name):
        boost_model = self.model.get_booster()
        mlflow.xgboost.log_model(boost_model, name=model_name)

    # OPTIONAL: custom_func
    # Called after evaluation. 
    # Allows you to log custom plots, metrics, or artifacts to MLflow using the provided data.
    # Leave as `pass` if you don't need it.
    def custom_func(self, df_train, df_val, df_test, y_test, y_probs):
        shap_fig = shap_explanations(self.model, df_test[self.features], "XGBoost")
        mlflow.log_figure(shap_fig, "custom_plots/shap_summary_plot.png")
        plt.close(shap_fig)

## Step 2: Add it to the Factory

Open `src/model_training/models/factory.py` and add one `elif` block. The lazy-import means your new library is only imported when your model is actually selected. Note this cell doesn't currently work due to relative location compared to the actual model files.

In [ ]:
# src/model_training/models/factory.py

from model_training.models.base_model import BaseSepsisModel

MODEL_REGISTRY = {}

def get_model(model_name, config, model_params, features) -> BaseSepsisModel:
    if model_name == "xgboost" and "xgboost" not in MODEL_REGISTRY:
        from .xgboost_model import XGBoostWrapper
        MODEL_REGISTRY["xgboost"] = XGBoostWrapper
    elif model_name == "lightgbm" and "lightgbm" not in MODEL_REGISTRY:
        from .lightgbm_model import LightGBMWrapper
        MODEL_REGISTRY["lightgbm"] = LightGBMWrapper
    elif model_name == "lstm" and "lstm" not in MODEL_REGISTRY:
        from .lstm_model import LSTMModelWrapper
        MODEL_REGISTRY["lstm"] = LSTMModelWrapper

    # --- ADD YOUR MODEL HERE (copy this block and adapt) ---
    # elif model_name == "your_model_name" and "your_model_name" not in MODEL_REGISTRY:
    #     from .your_model_file import YourModelWrapper
    #     MODEL_REGISTRY["your_model_name"] = YourModelWrapper
    # --------------------------------------------------------

    elif model_name not in MODEL_REGISTRY:
        raise ValueError(f"Model {model_name} not found in factory registry.")

    return MODEL_REGISTRY[model_name](config, model_params, features)

## Step 3: Add a Config Section

Open `src/model_training/config.yaml` and make two changes:

**1. Set `experiment.active_model`** to your model name (must match the string you used in the factory).

**2. Add a params section** for your model's hyperparameters. For tabular models with Optuna HPO, follow the XGBoost pattern with a `search_space` block. Each entry is `[min, max, type]` where `type` is one of `"float"`, `"int"`, `"log"`, or `"int_log"`. For simpler models without HPO, just list the params directly (your wrapper reads them via `self.model_params`).

```yaml
experiment:
  active_model: "your_model_name"   # must match the factory key exactly

# Example: model with Optuna HPO
tabular:
  models:
    your_model_name:
      search_space:
        learning_rate: [0.005, 0.05, "log"]
        max_depth: [2, 6, "int"]

# Example: model without HPO (just list params directly)
your_model_name:
  n_estimators: 200
  max_depth: 10
```

Then run as normal:

```bash
uv run src/model_training/main.py
```